## Imports


In [1]:
import polars as pl
from pathlib import Path

# Data Path


In [2]:
data_folder = Path("../data")

# Load Dataset


In [3]:
sales_df = pl.read_parquet(
    data_folder / "sales_cleaned.parquet"
)

# Preview Dataset
sales_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499


## Customer Metrics


In [5]:
customer_metrics = sales_df.group_by("customer_id").agg(
    pl.count().alias("total_orders"),
    pl.col("revenue").sum().alias("total_spent"),
    pl.col("revenue").mean().alias("avg_order_value"),
    pl.col("purchase_date").min().alias("first_purchase"),
    pl.col("purchase_date").max().alias("last_purchase")
)

customer_metrics

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_18920\363853550.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("total_orders"),


customer_id,total_orders,total_spent,avg_order_value,first_purchase,last_purchase
str,u32,i64,f64,datetime[μs],datetime[μs]
"""c721""",48,15702,327.125,2024-01-12 00:00:00,2024-12-14 00:00:00
"""c277""",74,23726,320.621622,2024-01-10 00:00:00,2024-12-24 00:00:00
"""c663""",32,10118,316.1875,2024-01-04 00:00:00,2024-12-30 00:00:00
"""c618""",37,11813,319.27027,2024-01-01 00:00:00,2024-12-26 00:00:00
"""c272""",64,19886,310.71875,2024-01-01 00:00:00,2024-12-27 00:00:00
…,…,…,…,…,…
"""c195""",53,17197,324.471698,2024-02-05 00:00:00,2024-12-30 00:00:00
"""c962""",64,21286,332.59375,2024-01-02 00:00:00,2024-12-18 00:00:00
"""c335""",40,13560,339.0,2024-01-06 00:00:00,2024-12-25 00:00:00


## Customer Lifetime


In [ ]:
customer_metrics = customer_metrics.with_columns(
    (
        pl.col("last_purchase") - pl.col("first_purchase")
    ).dt.total_days().alias("customer_lifetime_days")
)

# Purchase Frequency


In [7]:
customer_metrics = customer_metrics.with_columns(
    (
        pl.col("total_orders") / (pl.col("customer_lifetime_days") + 1)
    ).alias("purchase_frequency")
)

## Customer Segments


In [ ]:
customer_metrics = customer_metrics.with_columns(
    pl.when(pl.col("total_spent") > 20000)
    .then(pl.lit("VIP"))

    .when(pl.col("total_spent") > 10000)
    .then(pl.lit("Loyal"))

    .when(pl.col("total_spent") > 5000)
    .then(pl.lit("Regular"))

    .otherwise(pl.lit("Low Value"))
    .alias("customer_type")
)

# Segment Distribution


In [10]:
segment_distribution = customer_metrics.group_by("customer_type").agg(
    pl.count().alias("customers"),
    pl.col("total_spent").sum().alias("segment_revenue"),
).sort("segment_revenue", descending=True)

# Adding revenue_share_pct column

segment_distribution = segment_distribution.with_columns(

    (
        pl.col("segment_revenue")/pl.col("segment_revenue").sum() * 100

    ).round().alias("revenue_share_pct")

)

segment_distribution

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_18920\2154365367.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("customers"),


customer_type,customers,segment_revenue,revenue_share_pct
str,u32,i64,f64
"""Loyal""",629,9589136,59.0
"""VIP""",232,5612876,35.0
"""Regular""",117,953773,6.0
"""Low Value""",22,90165,1.0


Save the dataset


In [11]:
customer_metrics.write_parquet(data_folder/"customer_metrics.parquet")